### 🤖 Chat with PDF

In [ ]:
import os
import sys
import json
from pypdf import PdfReader
import dotenv
from openai import OpenAI
import chromadb
from IPython.display import display, Markdown

# LLMRouter handles the generation step; OpenAI client handles embeddings only
sys.path.insert(0, os.path.abspath("../.."))
from garage_helper import LLMRouter

dotenv.load_dotenv()

### ⚙️ Configuration

In [ ]:
PDF_FILE_PATH         = "../../data/02-RAG_Systems/simple_rag/Classic_Airent-3.pdf"
CHROMA_COLLECTION_NAME = "datasheet_rag"

# Embedding model — stays on Azure OpenAI (Bedrock doesn't serve text-embedding-3-small)
OPENAI_KEY      = os.getenv("OPENAI_API_KEY")
OPENAI_ENDPOINT = os.getenv("OPENAI_ENDPOINT")
EMBEDDING_MODEL = "text-embedding-3-small"

# Chat model — routed through LLMRouter → AWS Bedrock (Claude Haiku)
# Swap MODEL below to redirect to any provider without touching anything else
MODEL = "arn:aws:bedrock:us-east-1:791532114280:application-inference-profile/ry2uzni39tf7"

### 🚀 Initiating OpenAI Client & Chroma DB (In-Memory)

In [ ]:
# Embedding client — Azure OpenAI only (used for vector search, not generation)
embed_client = OpenAI(base_url=OPENAI_ENDPOINT, api_key=OPENAI_KEY)

# LLM router — all generation calls go through here → Bedrock Claude
# verbose=True prints each request + token count inline in the cell
router = LLMRouter(verbose=False)

chroma_client = chromadb.Client()
collection    = chroma_client.create_collection(name=CHROMA_COLLECTION_NAME)

print("Clients ready.")
print(f"  Embeddings : Azure OpenAI ({EMBEDDING_MODEL})")
print(f"  Generation : LLMRouter → Bedrock ({MODEL})")
print()
print("How the router works for this notebook:")
print("  router.generate(prompt, model=MODEL, system=...) calls BedrockClaudeProvider.generate()")
print("  which serialises the request as Bedrock Messages API JSON and calls boto3 invoke_model.")
print("  The reply text is returned as a plain string — no SDK-specific parsing needed.")

### 📚 Helper Functions

In [ ]:
def get_embedding(text: str) -> list:
    text = text.replace("\n", " ")
    response = embed_client.embeddings.create(input=[text], model=EMBEDDING_MODEL)
    return response.data[0].embedding

In [23]:
len(get_embedding("Who is the current chief minister"))

1536

In [9]:
def split_sections(text: str):
    sections = []
    current_header = None
    current_lines = []

    for line in text.splitlines():
        if line.strip().endswith(":"):  # header line
            # save previous section
            if current_header is not None:
                sections.append({
                    "header": current_header.replace(":",""),
                    "content": "\n".join(current_lines).strip().replace("\uf0b7", "")
                })
            # start new section
            current_header = line.strip()
            current_lines = []
        else:
            current_lines.append(line)

    # last section
    if current_header is not None:
        sections.append({
            "header": current_header,
            "content": "\n".join(current_lines).strip().replace("\uf0b7", "")
            
        })

    return sections

### 1️⃣  Loading PDF...

In [ ]:
reader = PdfReader(PDF_FILE_PATH)
full_text = ""
for page in reader.pages:
    full_text += page.extract_text()

In [ ]:
(full_text)

### ✂️  Chunking Text...

In [ ]:
sections = split_sections(full_text)
product_name = PDF_FILE_PATH.split('/')[-1].split('.')[0].replace('_', ' ')
print(f"Total sections in {product_name}: {len(sections)}")
for section in sections:
    header = section['header']
    content = section['content']
    print(f"{header} - Length of the content: {len(content)}")

chunks = [f"{sect['header']} of {product_name}:\n{sect['content']}" for sect in sections if sect['content'] != '']

Total sections in foss: 50
questions were identified. These are - Length of the content: 7681
adoption at KITE - Length of the content: 1233
identified - Length of the content: 1844
include - Length of the content: 437
3. AePS (Aadhaar Enabled Payment System) - Length of the content: 1363
driven by the following factors - Length of the content: 1530
adopting FOSS - Length of the content: 956
include - Length of the content: 5576
benefits from adopting FOSS solutions - Length of the content: 6811
are listed below - Length of the content: 1485
major benefits that Razorpay derives from FOSS - Length of the content: 4010
from adopting FOSS solutions - Length of the content: 555
that FOSS adoption brings with it some challenges - Length of the content: 4967
from adopting open source solutions - Length of the content: 3779
the following benefits using FOSS - Length of the content: 4141
experienced the following benefits of FOSS - Length of the content: 2936
Students satisfying any one of the

### 💾 Generating Embeddings & Storing...

In [64]:
ids = [str(i) for i in range(len(new_chunks))]
embeddings = []

# Loop through chunks and generate embeddings (Batching is better for production)
for i, chunk in enumerate(new_chunks):
    vec = get_embedding(chunk)
    embeddings.append(vec)
    if i % 5 == 0: print(f"   -> Processed {i+1}/{len(new_chunks)} chunks...", end="\r")


In [65]:
collection.add(
    documents=new_chunks,
    embeddings=embeddings,
    ids=ids
)
print("\n   -> Indexing complete!")


   -> Indexing complete!


### 🧠 RETRIEVAL & GENERATION LOOP

In [ ]:
user_queries = [
    "What is the packing variants of airent -3?",
    "What is the dosing of classic airent 3 needed for 25 kg cement?",
    "How to use classic airent?",
]
rag_system_prompt = """You are a helpful assistant. Use the provided context to answer the question.
    If the answer is not in the context, say you don't know."""
    

common_system_prompt = """You are a helpful assistant who has vast experience in construction and construction chemical field. Using your knowledge, answer the question."""


In [95]:
for ch in new_chunks:
    if "KITE".lower() in ch.lower():
        print(ch)
        break

questions were identified. These are of foss:
1. How and to what extent are organisations 
using FOSS?
2. What are the benefits (tangible and non-
tangible) they experience by virtue of adopting 
FOSS?
3. What are the challenges of working with FOSS?
4. What are the factors behind the organisation’s 
choice of software?
5. What potential legal and policy measures can 
support and promote FOSS in India?
As indicated earlier, within our mixed methods 
research framework, we adopted the case study 
approach to comprehensively address these research 
questions. To build methodologically rigorous case 
studies, we prepared a detailed, semi-structured 
questionnaire.
While it would have been preferable to build case 
studies from all sectors, we had to limit our case 
studies to four sectors (healthcare, education, 
finance and software and IT services) due to 
time and resource constraints. However, efforts 
were made to ensure greater diversity by trying 
to have four categories in each of

In [ ]:
rag_system_prompt    = "You are a helpful assistant. Use the provided context to answer the question. If the answer is not in the context, say you don't know."
common_system_prompt = "You are a helpful assistant who has vast experience in construction and construction chemical field. Using your knowledge, answer the question."

for query in user_queries:
    query_vec = get_embedding(query)
    results   = collection.query(query_embeddings=[query_vec], n_results=3)
    retrieved_context = "\n\n".join(results['documents'][0])

    print(f"\nQuery: {query}")
    print(f"Retrieved Context:\n{retrieved_context[:300]}...")

    rag_user_message    = f"Context:\n{retrieved_context}\n\nQuestion:\n{query}"
    common_user_message = f"Question: {query}"

    # Both calls go through the router → Bedrock Claude
    rag_reply    = router.generate(rag_user_message,    model=MODEL, system=rag_system_prompt,    max_tokens=400)
    common_reply = router.generate(common_user_message, model=MODEL, system=common_system_prompt, max_tokens=400)

    print(f"\n🤖 RAG answer    : {rag_reply}")
    print(f"\n🤖 General answer: {common_reply}")
    print("-" * 70)